In [1]:
# MUHAMMAD ZAID AQIL BIN ARIZAL (SW01083598) SECTION 01AT
# MUHAMMAD HAZIQ BIN ROHAIZAD (SW01083541) SECTION 01AT

import requests
import time
from bs4 import BeautifulSoup
import csv
# Why need Selenium? because Lazada load their review content with JavaScript, not initial HTML
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [2]:
# Function to scrape reviews from Lazada review page (current DOM)
# Each review is in div.item; name=span.reviewer, date=span.time, text=item-content-main-content-reviews-item

def scrape_page(soup, reviews):
    for review_block in soup.find_all('div', class_='item'):
        # Reviewer name
        name_el = review_block.find('span', class_='reviewer')
        name = name_el.get_text(strip=True) if name_el else ''
        # Review date
        date_el = review_block.find('span', class_='time')
        date = date_el.get_text(strip=True) if date_el else ''
        # Review text
        text_container = review_block.find('div', class_='item-content-main-content-reviews-item')
        text = text_container.get_text(strip=True) if text_container else ''
        reviews.append({'Text': text, 'Author': name, 'Date': date})

In [3]:
# Base URL and headers
base_url = 'https://www.lazada.com.my/products/pdp-i1035622558-s2790798853.html?spm=a2o4k.10415192.0.0.3ff91bf9rz4xDX&search=store&mp=3'
headers = {'User-Agent': 'Mozilla/5.0'}

In [4]:
# List to store reviews
reviews = []

In [5]:
# Scrape first 5 pages of Lazada reviews using Selenium pagination
def scrape_all_pages(url, max_pages=6):
    driver = webdriver.Chrome()
    try:
        driver.get(url)

        current_page = 1
        while current_page <= max_pages:
            # Wait until review items are present on the page
            WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CLASS_NAME, 'item'))
            )

            # Parse current page
            soup = BeautifulSoup(driver.page_source, 'html.parser')
            scrape_page(soup, reviews)

            # Stop if we've reached the target page count
            if current_page == max_pages:
                break

            # Find and click the "Next Page" button
            try:
                next_btn = WebDriverWait(driver, 10).until(
                    EC.element_to_be_clickable((By.CLASS_NAME, 'iweb-pagination-next'))
                )
                next_btn.click()
                current_page += 1
            except Exception:
                # No more pages or button not clickable
                break
    finally:
        driver.quit()

# 1. Selenium opens a real browser (Chrome).
# 2. The browser loads the page and runs the JavaScript.
# 3. The JavaScript fetches review data (often from an API) and inserts it into the page.
# 4. After that, the DOM contains the div.review-item elements.
# 5. Selenium gives you driver.page_source the fully rendered HTML.
# 6. BeautifulSoup parses that and finds the reviews.


In [6]:
# Scrape reviews from all pages
scrape_all_pages(base_url, max_pages=6)

TimeoutException: Message: 
Stacktrace:
0   chromedriver                        0x00000001025e96b4 cxxbridge1$str$ptr + 3127600
1   chromedriver                        0x00000001025e1a50 cxxbridge1$str$ptr + 3095756
2   chromedriver                        0x00000001020be56c _RNvCsdExgN8vFLbb_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 75432
3   chromedriver                        0x0000000102107864 _RNvCsdExgN8vFLbb_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 375200
4   chromedriver                        0x0000000102146620 _RNvCsdExgN8vFLbb_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 632668
5   chromedriver                        0x00000001020fbb9c _RNvCsdExgN8vFLbb_7___rustc35___rust_no_alloc_shim_is_unstable_v2 + 326872
6   chromedriver                        0x00000001025a8680 cxxbridge1$str$ptr + 2861308
7   chromedriver                        0x00000001025abdd4 cxxbridge1$str$ptr + 2875472
8   chromedriver                        0x000000010258da7c cxxbridge1$str

In [ ]:
# Save quotes to CSV file
with open('review-lazada-extracted.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=['Text', 'Author', 'Date'])
    writer.writeheader()
    writer.writerows(reviews)